# First Regression/Classification Model

#### Proposed pipeline
`EMG + IMU → preprocessing/features → regression proxy (0–50) → discrete state → bilateral transition validation → accepted motor state`

The data contains:
- 8 EMG channels
- 6 IMUs (left/right thigh, shank, foot)
- each IMU has Accel X/Y/Z and Gyro X/Y/Z
- standing baseline + five-step walking trial

Time/history is used for **state-transition tracking and fault detection**, not future prediction.

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, confusion_matrix

plt.rcParams['figure.figsize'] = (12, 4)
pd.set_option('display.max_columns', 60)
RANDOM_STATE = 42

Matplotlib is building the font cache; this may take a moment.


## 1. Load the two recordings

In [ ]:
STANDING_FILE = "Data\\standing still z.xlsx"
WALKING_FILE = "Data\\Walking z 1.xlsx"

standing = pd.read_excel(STANDING_FILE)
walking = pd.read_excel(WALKING_FILE)

print("Standing:", standing.shape)
print("Walking :", walking.shape)
display(walking.head())

## 2. Identify signal groups automatically

In [ ]:
time_col = 'Time_s'
emg_cols = list(walking.columns[1:9])
imu_cols = [c for c in walking.columns if c not in [time_col] + emg_cols]
left_cols = [c for c in imu_cols if '_L_' in c]
right_cols = [c for c in imu_cols if '_R_' in c]

print(f"EMG channels: {len(emg_cols)}")
print(f"IMU columns : {len(imu_cols)}")
print(f"Left IMU    : {len(left_cols)}")
print(f"Right IMU   : {len(right_cols)}")
print("
EMG:", emg_cols)

## 3. Basic data-quality analysis
Check missing values, duplicates, sampling interval, and simple descriptive statistics before modeling.

In [ ]:
def quality_report(df, name):
    dt = df[time_col].diff().dropna()
    print(f"--- {name} ---")
    print("Rows / columns:", df.shape)
    print("Missing cells:", int(df.isna().sum().sum()))
    print("Duplicate rows:", int(df.duplicated().sum()))
    print("Median Δt:", dt.median(), "s")
    if dt.median() > 0:
        print("Approx sampling rate:", round(1/dt.median(), 2), "Hz")

quality_report(standing, "Standing baseline")
quality_report(walking, "Walking")

## 4. Visualize EMG: standing vs walking
To keep plots readable, the code shows a short interval. Change `PLOT_SECONDS` as needed.

In [ ]:
PLOT_SECONDS = 2.0

def plot_channels(df, cols, title, seconds=PLOT_SECONDS):
    part = df[df[time_col] <= df[time_col].iloc[0] + seconds]
    fig, ax = plt.subplots(figsize=(14, 6))
    for c in cols:
        ax.plot(part[time_col], part[c], label=c, linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Signal value")
    ax.legend(ncol=2, fontsize=8)
    ax.grid(alpha=0.2)
    plt.show()

plot_channels(standing, emg_cols, "EMG — standing baseline")
plot_channels(walking, emg_cols, "EMG — walking")

## 5. Visualize bilateral thigh/shank/foot motion
Gyroscope signals are especially useful for seeing movement direction and repeated gait patterns.

In [ ]:
for segment in ['Thigh', 'Shank', 'Foot']:
    cols = [f'{segment}_L_Gyro_X', f'{segment}_L_Gyro_Y', f'{segment}_L_Gyro_Z',
            f'{segment}_R_Gyro_X', f'{segment}_R_Gyro_Y', f'{segment}_R_Gyro_Z']
    plot_channels(walking, cols, f"{segment} gyroscope — left vs right", seconds=walking[time_col].iloc[-1])

## 6. Baseline normalization
Subtract the mean standing value from every sensor channel. This treats standing as the neutral reference and makes walking values represent deviation from baseline.

In [ ]:
signal_cols = emg_cols + imu_cols
baseline_mean = standing[signal_cols].mean()

walking_zeroed = walking.copy()
walking_zeroed[signal_cols] = walking[signal_cols] - baseline_mean

display(walking_zeroed.head())

## 7. Temporary bilateral 0–50 regression targets
**Prototype only.** Because true hip angle is unavailable, we estimate a thigh tilt proxy from accelerometer orientation relative to the standing baseline, then scale observed walking excursion to 0–50.

This lets us test the software architecture, but it must not be reported as validated anatomical hip angle.

In [ ]:
def accel_tilt_deg(df, side):
    # Generic tilt magnitude from accelerometer. Axis convention must be verified for the real sensor setup.
    x = df[f'Thigh_{side}_Accel_X'].to_numpy()
    y = df[f'Thigh_{side}_Accel_Y'].to_numpy()
    z = df[f'Thigh_{side}_Accel_Z'].to_numpy()
    return np.degrees(np.arctan2(np.sqrt(x*x + y*y), z))

# Neutral orientation from standing baseline
neutral_L = np.nanmedian(accel_tilt_deg(standing, 'L'))
neutral_R = np.nanmedian(accel_tilt_deg(standing, 'R'))

walking_work = walking_zeroed.copy()
# Use RAW acceleration for orientation, then reference it to standing.
raw_tilt_L = accel_tilt_deg(walking, 'L')
raw_tilt_R = accel_tilt_deg(walking, 'R')

walking_work['TiltProxy_L_deg'] = np.abs(raw_tilt_L - neutral_L)
walking_work['TiltProxy_R_deg'] = np.abs(raw_tilt_R - neutral_R)

def scale_to_0_50(x):
    # Robust scaling: 95th percentile is treated as the observed high excursion.
    high = np.nanpercentile(x, 95)
    if high <= 0:
        return np.zeros_like(x)
    return np.clip(50.0 * x / high, 0, 50)

walking_work['TargetProxy_L'] = scale_to_0_50(walking_work['TiltProxy_L_deg'].to_numpy())
walking_work['TargetProxy_R'] = scale_to_0_50(walking_work['TiltProxy_R_deg'].to_numpy())

plt.plot(walking_work[time_col], walking_work['TargetProxy_L'], label='Left proxy')
plt.plot(walking_work[time_col], walking_work['TargetProxy_R'], label='Right proxy')
plt.title('Temporary bilateral control-position proxy (0–50)')
plt.xlabel('Time (s)')
plt.ylabel('Proxy control value')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 8. Feature engineering
The model itself is not forecasting. Rolling features summarize the **current/recent sensor condition** and can be removed if the team wants strictly instantaneous inputs.

Here we use raw zeroed signals plus short rolling mean and standard deviation.

In [ ]:
WINDOW = 20  # samples; adjust after confirming sampling rate/control loop
feature_base = signal_cols

features = walking_zeroed[feature_base].copy()
for c in feature_base:
    features[c + '_mean'] = walking_zeroed[c].rolling(WINDOW, min_periods=1).mean()
    features[c + '_std'] = walking_zeroed[c].rolling(WINDOW, min_periods=1).std().fillna(0)

features = features.replace([np.inf, -np.inf], np.nan).fillna(0)
print("Feature matrix:", features.shape)

## 9. Starter regression model
A Random Forest is used because it is a simple nonlinear baseline for tabular sensor features. We train one model per side.

**Important:** random splitting is acceptable only for this first prototype. For final evaluation, split by separate trials/subjects to avoid leakage between highly similar neighboring samples.

In [ ]:
X = features
idx_train, idx_test = train_test_split(np.arange(len(X)), test_size=0.25, random_state=RANDOM_STATE)

models = {}
predictions = {}

for side in ['L', 'R']:
    y = walking_work[f'TargetProxy_{side}'].to_numpy()
    model = RandomForestRegressor(
        n_estimators=120,
        max_depth=12,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    model.fit(X.iloc[idx_train], y[idx_train])
    pred = np.clip(model.predict(X.iloc[idx_test]), 0, 50)
    models[side] = model
    predictions[side] = pred

    print(f"
{side} regression")
    print("MAE :", round(mean_absolute_error(y[idx_test], pred), 3))
    print("RMSE:", round(np.sqrt(mean_squared_error(y[idx_test], pred)), 3))
    print("R²  :", round(r2_score(y[idx_test], pred), 3))

## 10. Visualize regression output

In [ ]:
order = np.argsort(idx_test)
for side in ['L', 'R']:
    y = walking_work[f'TargetProxy_{side}'].to_numpy()
    plt.figure(figsize=(14,4))
    plt.plot(walking_work[time_col].iloc[idx_test[order]], y[idx_test[order]], label='Proxy target', linewidth=1)
    plt.plot(walking_work[time_col].iloc[idx_test[order]], predictions[side][order], label='Regression output', linewidth=1)
    plt.title(f'{side} side — regression proxy target vs output')
    plt.xlabel('Time (s)')
    plt.ylabel('0–50 control value')
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

## 11. Convert regression output into discrete motor states
Ten 5-unit bins are used initially. This is **deterministic state mapping**, not a second ML model.

- S0: [0,5)
- S1: [5,10)
- ...
- S9: [45,50]

In [ ]:
def value_to_state(value, step=5, max_state=9):
    value = np.clip(value, 0, 50)
    return int(min(value // step, max_state))

# Generate predictions for the complete walking sequence for transition analysis.
for side in ['L', 'R']:
    walking_work[f'PredValue_{side}'] = np.clip(models[side].predict(X), 0, 50)
    walking_work[f'State_{side}'] = walking_work[f'PredValue_{side}'].apply(value_to_state)

walking_work[[time_col, 'PredValue_L', 'State_L', 'PredValue_R', 'State_R']].head(20)

## 12. Visualize left/right state progression
This is the important time-series use in the proposed design: **observe how accepted states change over time**, rather than predict a future state.

In [ ]:
plt.figure(figsize=(14,5))
plt.step(walking_work[time_col], walking_work['State_L'], where='post', label='Left state')
plt.step(walking_work[time_col], walking_work['State_R'], where='post', label='Right state')
plt.yticks(range(10), [f'S{i}' for i in range(10)])
plt.xlabel('Time (s)')
plt.ylabel('Control state')
plt.title('Bilateral state transitions')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 13. Bilateral transition / fault-detection prototype
The validator considers **both hips together**. A candidate state can be rejected when:
1. either side jumps too many states in one control update;
2. both sides simultaneously make a large forward jump;
3. optional rapid direction reversal occurs.

The thresholds below are placeholders and should later be justified using the real gait data and the motor/control update interval.

In [ ]:
MAX_STATE_JUMP = 2
BILATERAL_LARGE_FORWARD = 2

def transition_check(prev_l, prev_r, curr_l, curr_r):
    dl = curr_l - prev_l
    dr = curr_r - prev_r
    reasons = []

    if abs(dl) > MAX_STATE_JUMP:
        reasons.append(f'left jump {dl:+d}')
    if abs(dr) > MAX_STATE_JUMP:
        reasons.append(f'right jump {dr:+d}')

    # Bilateral coordination safeguard: both sides should not suddenly demand
    # a large forward advance together.
    if dl >= BILATERAL_LARGE_FORWARD and dr >= BILATERAL_LARGE_FORWARD:
        reasons.append('simultaneous large bilateral forward jump')

    return len(reasons) == 0, '; '.join(reasons)

accepted_l = [int(walking_work['State_L'].iloc[0])]
accepted_r = [int(walking_work['State_R'].iloc[0])]
faults = [False]
reasons = ['']

for i in range(1, len(walking_work)):
    cand_l = int(walking_work['State_L'].iloc[i])
    cand_r = int(walking_work['State_R'].iloc[i])
    ok, reason = transition_check(accepted_l[-1], accepted_r[-1], cand_l, cand_r)

    if ok:
        accepted_l.append(cand_l)
        accepted_r.append(cand_r)
        faults.append(False)
        reasons.append('')
    else:
        # Safe prototype behavior: hold the last accepted bilateral state.
        accepted_l.append(accepted_l[-1])
        accepted_r.append(accepted_r[-1])
        faults.append(True)
        reasons.append(reason)

walking_work['AcceptedState_L'] = accepted_l
walking_work['AcceptedState_R'] = accepted_r
walking_work['Fault'] = faults
walking_work['FaultReason'] = reasons

print('Flagged transitions:', int(walking_work['Fault'].sum()))
display(walking_work.loc[walking_work['Fault'],
    [time_col, 'State_L','State_R','AcceptedState_L','AcceptedState_R','FaultReason']].head(20))

## 14. Visualize accepted states and faults

In [ ]:
plt.figure(figsize=(14,5))
plt.step(walking_work[time_col], walking_work['AcceptedState_L'], where='post', label='Accepted left')
plt.step(walking_work[time_col], walking_work['AcceptedState_R'], where='post', label='Accepted right')

fault_rows = walking_work[walking_work['Fault']]
if len(fault_rows):
    plt.scatter(fault_rows[time_col], fault_rows['AcceptedState_L'], marker='x', label='Flagged transition')

plt.yticks(range(10), [f'S{i}' for i in range(10)])
plt.xlabel('Time (s)')
plt.ylabel('Accepted state')
plt.title('Bilateral accepted state sequence with transition faults')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 15. Transition matrix — learn what the recorded gait actually does
Instead of inventing every valid bilateral transition, count which `(Left state, Right state)` pairs and transitions appear in the walking trial. With more trials, this can become the basis for a data-driven allowed-transition table.

In [ ]:
pairs = list(zip(walking_work['AcceptedState_L'], walking_work['AcceptedState_R']))
transition_counts = {}
for a, b in zip(pairs[:-1], pairs[1:]):
    key = (a, b)
    transition_counts[key] = transition_counts.get(key, 0) + 1

transition_table = pd.DataFrame([
    {'From_L': a[0], 'From_R': a[1], 'To_L': b[0], 'To_R': b[1], 'Count': count}
    for (a, b), count in transition_counts.items()
]).sort_values('Count', ascending=False)

display(transition_table.head(25))

## 16. Optional state-classification evaluation
Because the regression output is ultimately quantized, evaluate whether the predicted proxy and target proxy land in the same state. This is often more meaningful for the motor controller than angle/proxy error alone.

In [ ]:
for side in ['L','R']:
    true_states = walking_work[f'TargetProxy_{side}'].apply(value_to_state)
    pred_states = walking_work[f'PredValue_{side}'].apply(value_to_state)
    print(side, 'state agreement:', round(accuracy_score(true_states, pred_states), 4))

## 17. Feature importance — initial analysis only
This can help identify which EMG/IMU measurements the Random Forest uses most. Feature importance does **not** prove physiological causality.

In [ ]:
for side in ['L','R']:
    imp = pd.Series(models[side].feature_importances_, index=X.columns).sort_values(ascending=False).head(20)
    plt.figure(figsize=(10,6))
    imp.sort_values().plot(kind='barh')
    plt.title(f'Top Random Forest features — {side} proxy')
    plt.xlabel('Feature importance')
    plt.tight_layout()
    plt.show()

## 18. Export prototype results
This produces a CSV containing candidate/accepted states and fault flags for later inspection or motor-interface prototyping.

In [ ]:
output_cols = [time_col, 'PredValue_L','State_L','AcceptedState_L',
               'PredValue_R','State_R','AcceptedState_R','Fault','FaultReason']
walking_work[output_cols].to_csv('prototype_bilateral_states.csv', index=False)
print('Saved prototype_bilateral_states.csv')

# What should be improved next?

1. **Do not present `TargetProxy_L/R` as true hip angles.** They are placeholders created only because the current files lack ground-truth joint angle.
2. Verify IMU units, axis orientation, and sensor mounting before interpreting tilt physically.
3. Decide whether the final ML input should use all IMUs, EMG only, or selected features.
4. Analyze several walking trials/subjects before defining the final number of states.
5. Replace random train/test splitting with trial-wise or subject-wise splitting when more recordings exist.
6. Determine valid **bilateral** state combinations/transitions from repeated gait data rather than relying only on hand-written rules.
7. Add transition timing constraints (states/second or degrees/second equivalent) once the motor/control update rate is known.
8. Consider classifier baselines (Random Forest/XGBoost) if the final target becomes directly labeled discrete states rather than a continuous control variable.
9. Add a safe hold/neutral policy for invalid, noisy, or low-confidence classifications.

The architectural separation should remain:

`Sensor analysis → model output → state mapping → bilateral transition validator → fault handling → motor command`